In [1]:
# IFRS 9 — Python Translation 
# -----------------------------------------------------
# Steps:
# 1) Load data
# 2) Build synthetic EAD features (credit_limit_demo, ead_demo)
# 3) Apply skew transforms (log, sqrt)
# 4) Create month_key and build monthly macro table (means)
# 5) Create 3- and 6-month lags of macro variables and merge back
# 6) Filter rows where 6M lag exists (SAS: where gdp_l6 ^= .)
# 7) Logistic regression: default_flag ~ dpd + gdp_l3 + rate_l3 + unemp_l6
# 8) VIF for selected predictors
# 9) Simple VARCLUS analogue (correlation check)

import pandas as pd
import numpy as np
from pathlib import Path

# Stats/ML
import statsmodels.api as sm
from statsmodels.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# -----------------------------
# 0) Load data
# -----------------------------
# Set your CSV path here (change to your file if needed)
CSV_PATH = Path("C:/Users/customer/Desktop/MyVideos/UDEMY LESSONS/IFRS 9 Python/Module 3 Dataset/ifrs9_pit_pd_intro_final.csv")

df = pd.read_csv(CSV_PATH)

# Coerce any "*date*" columns to datetime (e.g., report_date)
for c in df.columns:
    if "date" in c.lower():
        try:
            df[c] = pd.to_datetime(df[c])
        except Exception:
            pass

# -----------------------------
# 1) Synthetic EAD features
#    SAS:
#      credit_limit_demo = 3000 + 15*internal_score
#      ead_demo          = credit_limit_demo * credit_utilization
# -----------------------------
df["credit_limit_demo"] = 3000 + 15 * df["internal_score"]
df["ead_demo"] = df["credit_limit_demo"] * df["credit_utilization"]

# -----------------------------
# 2) Skew transforms
#    SAS:
#      ead_log  = log(ead_demo + 1)
#      ead_sqrt = sqrt(max(ead_demo,0))
# -----------------------------
df["ead_log"] = np.log(df["ead_demo"].clip(lower=0) + 1.0)
df["ead_sqrt"] = np.sqrt(df["ead_demo"].clip(lower=0))

# -----------------------------
# 3) Month key for PIT joins & lags
#    SAS:
#      month_key = intnx('month', report_date, 0, 'b')
#      format yymmn6.
# -----------------------------
# “Beginning of month” equivalent:
df["month_key"] = df["report_date"].values.astype("datetime64[M]")
df["month_key"] = pd.to_datetime(df["month_key"])

# -----------------------------
# 4) Monthly macro table (means)
# -----------------------------
macro_cols = ["macro_gdp_growth", "macro_unemployment", "macro_interest_rate"]
for col in macro_cols:
    if col not in df.columns:
        df[col] = np.nan  # keep demo robust if a macro column is missing

monthly = (
    df.groupby("month_key")[macro_cols]
      .mean()
      .rename(columns={
          "macro_gdp_growth": "gdp_0",
          "macro_unemployment": "unemp_0",
          "macro_interest_rate": "rate_0"
      })
      .reset_index()
)

monthly = monthly.sort_values("month_key").reset_index(drop=True)

# -----------------------------
# 5) Create 3- and 6-month lags
#    SAS uses self-join with intnx(); here we shift by rows
# -----------------------------
monthly["gdp_l3"]   = monthly["gdp_0"].shift(3)
monthly["unemp_l3"] = monthly["unemp_0"].shift(3)
monthly["rate_l3"]  = monthly["rate_0"].shift(3)

monthly["gdp_l6"]   = monthly["gdp_0"].shift(6)
monthly["unemp_l6"] = monthly["unemp_0"].shift(6)
monthly["rate_l6"]  = monthly["rate_0"].shift(6)

# Merge back to observations (SAS: left join to stage2)
dfm = df.merge(monthly, on="month_key", how="left")

# -----------------------------
# 6) Filter where 6M lag exists
#    SAS: where gdp_l6 ^= .
# -----------------------------
dfm = dfm[dfm["gdp_l6"].notna()].copy()

# -----------------------------
# 7) Logistic regression
#    :
#      model default_flag (event="1") = dpd gdp_l3 rate_l3 unemp_l6
# -----------------------------
y = dfm["default_flag"].astype(int)
X = dfm[["dpd", "gdp_l3", "rate_l3", "unemp_l6"]].copy()

mask = X.notna().all(axis=1) & y.notna()
X, y = X.loc[mask], y.loc[mask]

Xc = add_constant(X)  # intercept
logit_model = sm.Logit(y, Xc).fit(disp=False)

print("\n=== Logistic Regression Summary ===")
print(logit_model.summary2())

# -----------------------------
# 8) VIF for selected predictors
#    
#      internal_score credit_utilization
# -----------------------------
vif_df = dfm[["internal_score", "credit_utilization"]].dropna().copy()
vif_X = add_constant(vif_df)

vif_table = []
for i in range(1, vif_X.shape[1]):  # skip intercept
    vif_table.append({
        "variable": vif_X.columns[i],
        "VIF": float(variance_inflation_factor(vif_X.values, i))
    })
vif_table = pd.DataFrame(vif_table)

print("\n=== VIF (internal_score & credit_utilization) ===")
print(vif_table.to_string(index=False))

# -----------------------------
# 9) Simple VARCLUS analogue
#    With two variables, correlation tells the story.
# -----------------------------
corr_val = float(vif_df["internal_score"].corr(vif_df["credit_utilization"]))
print("\n=== VARCLUS analogue (correlation) ===")
print(f"internal_score vs credit_utilization: corr = {corr_val:.4f}, |corr| = {abs(corr_val):.4f}")

# -----------------------------
# 10) MEANS-like summary for EAD features
# -----------------------------
def proc_means_like(frame, cols):
    rows = []
    for c in cols:
        s = frame[c]
        s_nona = s.dropna()
        rows.append({
            "variable": c,
            "n": int(s.shape[0]),
            "nmiss": int(s.isna().sum()),
            "mean": float(s_nona.mean()) if len(s_nona) else np.nan,
            "p25": float(s_nona.quantile(0.25)) if len(s_nona) else np.nan,
            "median": float(s_nona.median()) if len(s_nona) else np.nan,
            "p75": float(s_nona.quantile(0.75)) if len(s_nona) else np.nan,
            "max": float(s_nona.max()) if len(s_nona) else np.nan
        })
    return pd.DataFrame(rows)

summary = proc_means_like(dfm, ["ead_demo", "ead_log", "ead_sqrt"])
print("\n=== PROC MEANS — EAD features ===")
print(summary.to_string(index=False))

# -----------------------------
# 11) PRINT
# -----------------------------
print("\n=== First 50 rows — stage1 equivalent ===")
print(df.head(50).to_string(index=False))

print("\n=== First 40 rows — stage2m equivalent ===")
print(dfm.head(40).to_string(index=False))





=== Logistic Regression Summary ===
                          Results: Logit
Model:              Logit            Method:           MLE        
Dependent Variable: default_flag     Pseudo R-squared: 0.095      
Date:               2025-11-11 20:47 AIC:              5265.0963  
No. Observations:   4205             BIC:              5296.8165  
Df Model:           4                Log-Likelihood:   -2627.5    
Df Residuals:       4200             LL-Null:          -2904.3    
Converged:          1.0000           LLR p-value:      1.7375e-118
No. Iterations:     5.0000           Scale:            1.0000     
--------------------------------------------------------------------
             Coef.    Std.Err.      z      P>|z|     [0.025   0.975]
--------------------------------------------------------------------
const       -0.7498     2.8577   -0.2624   0.7930   -6.3508   4.8513
dpd          0.0139     0.0006   22.4086   0.0000    0.0127   0.0151
gdp_l3       0.2455     0.4789    0.5125 

In [2]:

df.head(10)

,loan_origination_date,report_date,default_date,loan_age_m,credit_utilization,dpd,internal_score,age,macro_gdp_growth,macro_unemployment,...,employment_status,marital_status,dependents,region,stage,credit_limit_demo,ead_demo,ead_log,ead_sqrt,month_key
0,2021-10-25,2024-03-01,NaT,29,0.42,99,551,36,4.33,4.51,...,Employed,Single,3,West,3,11265,4731.30,8.462167,68.784446,2024-03-01
1,2018-06-11,2022-05-01,2022-08-10,47,1.00,139,868,66,0.29,6.18,...,Employed,Married,0,West,3,16020,16020.00,9.681656,126.570139,2022-05-01
2,2021-02-20,2024-08-01,NaT,42,0.43,27,351,57,2.29,4.14,...,Employed,Divorced,0,South,2,8265,3553.95,8.176096,59.615015,2024-08-01
3,2017-12-22,2024-09-01,NaT,81,1.00,154,755,21,2.50,6.88,...,Employed,Married,3,South,3,14325,14325.00,9.569831,119.687092,2024-09-01
4,2021-12-04,2022-07-01,2022-10-25,7,0.73,119,802,38,3.31,4.32,...,Employed,Married,2,West,3,15030,10971.90,9.303184,104.746838,2022-07-01
5,2017-03-22,2024-01-01,NaT,82,0.82,5,498,28,2.53,6.05,...,Employed,Divorced,2,North,2,10470,8585.40,9.057935,92.657434,2024-01-01
6,2018-11-08,2023-11-01,NaT,60,0.91,147,561,37,1.76,4.58,...,Employed,Married,3,East,3,11415,10387.65,9.248469,101.919821,2023-11-01
7,2019-02-25,2022-04-01,NaT,38,1.00,90,860,58,2.75,3.61,...,Self-Employed,Married,0,South,3,15900,15900.00,9.674137,126.095202,2022-04-01
8,2019-05-26,2022-04-01,2023-03-06,35,0.62,0,761,55,2.52,6.66,...,Employed,Married,1,North,2,14415,8937.30,9.098101,94.537294,2022-04-01
9,2016-07-21,2023-10-01,NaT,87,0.36,98,822,45,2.01,2.34,...,Self-Employed,Single,4,South,3,15330,5518.80,8.616097,74.288626,2023-10-01


In [6]:
dfm.head(40)

,loan_origination_date,report_date,default_date,loan_age_m,credit_utilization,dpd,internal_score,age,macro_gdp_growth,macro_unemployment,...,month_key,gdp_0,unemp_0,rate_0,gdp_l3,unemp_l3,rate_l3,gdp_l6,unemp_l6,rate_l6
0,2021-10-25,2024-03-01,NaT,29,0.42,99,551,36,4.33,4.51,...,2024-03-01,1.987357,5.034714,2.954643,1.909699,4.841579,2.937744,2.051176,5.047868,3.066765
2,2021-02-20,2024-08-01,NaT,42,0.43,27,351,57,2.29,4.14,...,2024-08-01,1.987259,4.796667,3.059111,1.908195,4.857218,3.009323,1.922168,4.918601,2.960909
3,2017-12-22,2024-09-01,NaT,81,1.00,154,755,21,2.50,6.88,...,2024-09-01,1.835390,5.081277,3.024255,2.028269,4.816026,3.031795,1.987357,5.034714,2.954643
4,2021-12-04,2022-07-01,2022-10-25,7,0.73,119,802,38,3.31,4.32,...,2022-07-01,2.001059,4.976765,2.992294,1.880276,5.123448,2.997310,1.975905,5.130190,3.042286
5,2017-03-22,2024-01-01,NaT,82,0.82,5,498,28,2.53,6.05,...,2024-01-01,2.039643,5.135357,3.064000,2.054823,4.880993,2.966738,2.061500,4.992286,2.978714
6,2018-11-08,2023-11-01,NaT,60,0.91,147,561,37,1.76,4.58,...,2023-11-01,1.841346,5.105962,2.958077,2.077143,4.852929,3.022429,1.893417,4.956917,2.982250
9,2016-07-21,2023-10-01,NaT,87,0.36,98,822,45,2.01,2.34,...,2023-10-01,2.054823,4.880993,2.966738,2.061500,4.992286,2.978714,2.052479,5.202314,3.012645
10,2017-01-08,2022-11-01,NaT,70,0.44,2,566,22,1.60,4.04,...,2022-11-01,1.790705,5.063910,2.952500,1.946695,4.935593,2.875932,1.925724,4.890724,3.019671
11,2017-01-28,2024-11-01,NaT,94,1.00,51,761,52,1.66,1.97,...,2024-11-01,1.953082,5.117055,3.114589,1.987259,4.796667,3.059111,1.908195,4.857218,3.009323
12,2020-09-27,2024-12-01,2025-08-03,51,1.00,103,303,28,1.93,5.30,...,2024-12-01,2.084697,4.751591,2.965985,1.835390,5.081277,3.024255,2.028269,4.816026,3.031795
